In [1]:
# Local project setup for GitHub / local-machine execution
# The private dataset is intentionally excluded from this repository.
# Set MS_DATA_ROOT to the folder containing the dataset before running this notebook,
# or place the private dataset in a local ./MS directory.

import os

MS_DATA_ROOT = os.environ.get("MS_DATA_ROOT")
if MS_DATA_ROOT:
    print(f"MS_DATA_ROOT={MS_DATA_ROOT}")
else:
    print("MS_DATA_ROOT is unset; using the default local dataset location ./MS")

# Example:
# export MS_DATA_ROOT="/absolute/path/to/private/MS"


# MS Lesion Detection from Brain MRI — v6: OCR-Verified Patient Identity + Crop-Then-Mask Pipeline

**WINC Research Internship 2026 — Nile University**

**What changed since v1** (based on what the fresh re-check of `MS.zip` found):
1. **Burned-in PHI removed properly.** v1's `crop_to_brain` only cropped black borders — it did
   **not** remove the patient name / DOB / hospital name that DICOM export burns directly into
   the pixels (top-left, top-right, bottom-left, bottom-right corners, confirmed visually on
   both classes: e.g. `Gomhoriah Radiology Center` vs `Mansoura New General Hospital`). v2 masks
   those four corner regions before anything else.
2. **Patient IDs rebuilt from actual file content, not filenames.** 148 exact-duplicate clusters
   (330 files) were found via MD5 hashing — and 140 of those clusters span **different**
   `img-NNNNN-...` index numbers, meaning the same physical scan was exported twice under two
   different "patient" numbers. Trusting the filename index as the patient key (as v1 did) risks
   splitting one real patient across train and test. v2 uses union-find over exact-duplicate
   hashes to build canonical patient clusters first, then falls back to the filename scheme only
   for images with no duplicate.
3. **Fusion contribution added (Section 9), grounded in the literature** — see Section 0 for the
   full argument and citations.
4. **Honest new finding:** removing the burned text closed almost none of the shortcut (98.8% -> 98.3%
   trivial-classifier accuracy — see Section 4). This is reported as-is rather than hidden; it is
   itself consistent with a specific, citable finding in the multisite-MRI shortcut literature
   (Section 0, point 1) and changes the recommended framing of the paper's contribution.

**What changed in v4** (found from a real Kaggle GPU run of v3 that reported 100% accuracy
everywhere -- test set, patient-grouped CV, and even the plain 9-feature biomarker model):
5. **A single patient's 347-image cluster (65.7% of the MS class) was dominating whichever split
   it landed in** (a real run put it almost entirely in val: `val MS=362` vs `train MS=146` --
   more MS images in val than train). Section 6 now caps images per patient (12) *before*
   splitting, and Section 8/9's full-dataset analyses use the same capped manifest.
6. **The deep models had no dropout and fully unfroze in Phase 2,** so they could (and did)
   memorize a ~140-patient dataset within a single epoch of fine-tuning. Section 9 now adds
   dropout before the head, only unfreezes the last conv block (not the whole backbone) in
   Phase 2, and uses 10x stronger weight_decay there.
7. **A label-shuffle sanity check (Section 10.1)** is added: retrain on randomly permuted labels
   and confirm accuracy drops to ~chance. If it doesn't, the pipeline still leaks the label
   somewhere and no accuracy number from this notebook should be trusted yet.

**What changed in v6, after confirming with the actual data source (the radiology center) that
images are not duplicated -- the export just tiles multiple slices/orientations per real patient:**
8. **Patient identity is now OCR'd from the burned-in name+DOB on every image**, not inferred from
   the `img-NNNNN` filename index. That index resets independently inside each orientation folder,
   so the same index number in `normal_axial` and `normal_sagital` can belong (and, verified
   directly by OCR, does belong) to two completely different real patients -- which is what
   actually produced v3's implausible "one patient = 65.7% of the MS class" cluster. Section 3
   replaces the filename/duplicate-based union-find with real OCR'd identity, with per-file exact-
   duplicate linking kept only as a secondary safety net. This is slower (~10-15 min the first run,
   cached afterward) but is now checked against the source rather than assumed from a naming
   convention.
9. **Preprocessing fixed again, more carefully:** masking the burned-in text *before* computing the
   brain bounding box (v2/v3) turned out to crop MS and Normal images to different, class-correlated
   tightness (Normal shrank to 64% of frame, MS stayed at 100% -- a brand new shortcut). v6 computes
   the crop on the unmasked image first, then masks the corners of the already-cropped region. On
   the actual dataset this took the shortcut-audit accuracy from v3's 92.8% down to **77.5%** --
   real, substantial further progress, though still above chance (Section 5 explains why, and why
   pushing it lower risks tuning the pipeline to beat this specific test rather than genuinely
   fixing the data).

None of this is expected to bring accuracy down to some specific "good" number -- the point is
diagnostic, not cosmetic. Given Section 5's own finding (a ~92.7% non-anatomical shortcut
classifier survives the best preprocessing fix found so far), a properly-regularized, leakage-free
model may *still* land well above what a "fair" classifier should score on this dataset. That
outcome should be reported as-is, with the Section 5 audit as the explanation, rather than treated
as a bug to keep patching until the number looks better.

Sections 1–8 (audit, corner-mask + brain-crop + CLAHE, patient-cluster dedup, split, augmentation,
biomarkers) run on CPU. Sections 9–11 (deep models, fusion, Grad-CAM) need GPU — Kaggle/Colab.
`MAX_EPOCHS = 50` with early stopping everywhere.


## 0. Where the contribution is — the central finding, found late but decisive

**The headline finding of this notebook, discovered by OCR'ing the burned-in hospital name (not
just the patient name) on a real sample of the data: the class label is almost completely
confounded with acquisition site.** 95/100 sampled MS(FLAIR) images carry `MANSOURA NEW GENERAL
HOSPITAL` in their header; 100/100 sampled Normal images carry `Gomhoriah Radiology Center`
(Section 4.1). This is not a partial site imbalance of the kind Zech et al. (2018, *PLoS
Medicine*) or the 2023 JAMIA multisite-MRI study describe -- it is (within measurement/OCR error)
a **complete** confound: for essentially every image in this dataset, "which hospital is this
from" and "is this labeled MS or Normal" have the identical answer. That single fact governs how
every other result in this notebook must be interpreted, and is why it is presented first rather
than as a late finding buried in Discussion:

**1. No classifier, however designed, can distinguish "MS" from "acquisition site" on this
dataset, and this is a mathematical fact about the data, not a limitation of any particular model.**
A domain-adversarial approach (Ganin & Lempitsky, 2015) -- explicitly training a shared feature
extractor to be unable to predict domain/site while still predicting the true label, via a
gradient-reversal layer -- was considered as a next step and deliberately **not** implemented once
this was found: DANN only recovers a label-relevant, site-invariant representation when a domain
has examples of *both* classes, so the adversarial signal has something to push against. With a
100%-confounded design, "site-invariant" and "label-uninformative" collapse into the same
constraint -- there is no representation that is both predictive of the true label and blind to
site, because on this data those are the same variable. Attempting DANN here would not produce a
meaningfully debiased model; it would either fail to converge or destroy real signal along with
the site signal, and reporting a resulting accuracy number would be **more** misleading than
reporting none, not less. This reasoning -- not a failed experiment -- is the deliverable, and is
itself a citable methodological point: check for complete confounding *before* reaching for
debiasing techniques, because some confounds are not fixable by modeling at all.

**2. Every accuracy number elsewhere in this notebook (Sections 8-11) is consistent with EITHER
"the model learned to detect FLAIR lesions" OR "the model learned to detect which hospital's export
pipeline produced this image" -- and no experiment run so far, or plausible to run on this data
alone, can tell those two explanations apart.** This reframes the earlier staged shortcut-audit
work (Sections 4-5: 98.8% -> 74.7%) not as "we mostly fixed the confound" but as "we characterized
how much of the confound is attributable to text/contrast/framing artifacts specifically, while
the underlying site confound in the *labeling itself* remains total and is not fixable by image
preprocessing at all." The gap between the interpretable biomarker model (~90%, Section 8) and the
deep models (100%, Section 10) is now better read as: the deep model's much larger capacity finds
more of whatever separates the two sites (whether that's lesion pathology, scanner hardware, or
export software), not necessarily more of the disease signal specifically.

**3. What this notebook can still honestly claim as a contribution**, restated with this finding
front and center:
- **A documented, staged shortcut-audit methodology** (Sections 2-5) culminating in the discovery
  of complete site-label confounding -- a stronger, more specific instance of the general problem
  Zech et al. and the JAMIA paper describe, worth reporting as a case study of how far a confound
  can go undetected without this kind of check, and how even substantial, successful preprocessing
  fixes (86.7%-point drop, 98.8%->~75% on the non-anatomical shortcut classifier) do not touch a
  confound baked into the labeling process itself.
- **An interpretable radiomics/biomarker model (Section 8) as the more honestly-scoped result** --
  not because it is immune to the confound (it is not: nothing here is), but because its ~90%
  ceiling under patient-grouped CV, well below the deep models' literal 100%, is itself evidence
  that whatever it is keying on requires the model to actually process lesion-shaped structures
  within brain tissue, rather than whatever else (border style, contrast curve, compression
  artifacts) the unconstrained deep models can additionally exploit.
- **A clear, reproducible template for detecting complete confounding via burned-in metadata OCR**
  (Section 4.1) -- directly actionable for anyone auditing a similarly-sourced clinical imaging
  dataset before training on it.

**What this notebook cannot claim, and should not be written as claiming:** that any accuracy
number here demonstrates MS-lesion detection capability. The honest framing for the paper is a
methods/audit contribution about confound detection in a specific, real, multi-source clinical
dataset -- not a classification-performance paper. A future dataset with both classes represented
at both (or more) sites would be needed to make any disease-detection claim, and that limitation
should be stated as plainly in the paper as it is here.

## 1. Setup & reproducibility

In [2]:
import os, re, glob, hashlib, random
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import cv2
from scipy import ndimage

pd.set_option("display.max_colwidth", 120)
plt.rcParams["figure.dpi"] = 110
SEED = 42

def set_all_seeds(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except ImportError:
        pass

set_all_seeds(SEED)


## 2. Data inventory (recursive — v1's flat scan missed the nested `flair axial/flair axial/` folder)

In [3]:
_CANDIDATES = [
    os.environ.get("MS_DATA_ROOT"),
    "data/MS",
    "./MS",
    os.path.expanduser("~/MS"),
]
DATA_ROOT = next((c for c in _CANDIDATES if c and os.path.isdir(c)), "./MS")
print("DATA_ROOT =", DATA_ROOT)

CLASS_FOLDERS = {
    "flair_axial":   (os.path.join(DATA_ROOT, "flair axial"),   "MS (FLAIR)", "axial"),
    "flair_sagital": (os.path.join(DATA_ROOT, "flair sagital"), "MS (FLAIR)", "sagital"),
    "normal_axial":  (os.path.join(DATA_ROOT, "normal axial"),  "Normal",     "axial"),
    "normal_sagital":(os.path.join(DATA_ROOT, "normal sagital"),"Normal",     "sagital"),
}
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}

def scan_all_files():
    rows = []
    for key, (folder, label, plane) in CLASS_FOLDERS.items():
        if not os.path.isdir(folder):
            print(f"[WARN] not found: {folder}"); continue
        for root, dirs, files in os.walk(folder):
            for fn in files:
                ext = os.path.splitext(fn)[1].lower()
                fp = os.path.join(root, fn)
                rows.append(dict(source_key=key, class_label=label, plane=plane, filename=fn,
                                  filepath=fp, ext=ext, is_image_ext=ext in IMAGE_EXTS))
    return pd.DataFrame(rows)

raw_files = scan_all_files()
print(f"Total files found (recursive): {len(raw_files)}")
display(raw_files.groupby("class_label").size())


DATA_ROOT = ./MS
[WARN] not found: ./MS/flair axial
[WARN] not found: ./MS/flair sagital
[WARN] not found: ./MS/normal axial
[WARN] not found: ./MS/normal sagital
Total files found (recursive): 0


KeyError: 'class_label'

## 3. Patient identity via OCR on the burned-in header (not the filename index)

**Confirmed directly with the source radiology center: the images are not duplicated.** Their
export just tiles multiple slices/orientations per patient (top-to-bottom, or lengthwise), which
this dataset then splits across per-orientation folders. That explains something this notebook's
earlier versions got wrong in two different ways, both traced to the same root cause:

- v3's "347-image mega-cluster" (Section 3.1 in earlier versions) was **not** one patient with 347
  duplicate images -- it was an artifact of trusting `img-NNNNN-...` as a patient ID.
- The `img-NNNNN` index number **resets independently inside each orientation folder**
  (`normal_axial`'s numbering starts over from `normal_sagital`'s, etc.), so `img-00031-...` in
  `normal axial` and `img-00031-...` in `normal sagital` are, in general, two *different* real
  patients who happen to share an index number. Verified directly with OCR on the burned-in header
  text of the actual files: `img-00031-...` in `normal_axial` reads `SAMEIR MOH. ELBAYOUMI, 36Y,
  b.1986`; the same index number in `normal_sagital` reads `SAMIER ASAD MOHAMED, 31Y, b.1991` --
  two unrelated people. The earlier union-find (which unioned by filename index first, then by
  exact-duplicate content) inherited this bug wherever two different real patients' images
  happened to share an index number, merging them.

**The real fix:** since the patient's actual name and date of birth are burned directly into the
image pixels (top-left corner, confirmed present on effectively every image, both classes), OCR
that text and use `(normalized name, DOB)` as the patient key instead of any filename convention.
This is slower (~0.3-0.4s/image, so budget ~10-15 minutes for the full dataset the first time --
cache the result to disk so it only has to run once) but it is the only patient identifier in this
dataset that is actually verified against the source, rather than inferred from an export
convention that turned out not to hold across folders.

In [ ]:
# pip install pytesseract if not already available (tesseract-ocr binary must also be on PATH)
import pytesseract
import re as _re_ocr

OCR_CACHE_PATH = "patient_ocr_cache.csv"  # cache result so the OCR step only runs once per local session

def ocr_header_text(filepath, w_frac=0.42, h_frac=0.14):
    """OCRs the top-left corner (patient name + DOB line), upscaled 3x for OCR accuracy."""
    with Image.open(filepath) as im:
        im = im.convert("L")
        w, h = im.size
        crop = im.crop((0, 0, int(w * w_frac), int(h * h_frac)))
        crop = crop.resize((crop.width * 3, crop.height * 3), Image.LANCZOS)
        return pytesseract.image_to_string(crop).strip()

def normalize_ocr_to_patient_key(raw_text):
    """Turns raw OCR text into a (name, DOB) key. Returns None if no usable name was read --
    those images fall back to the filename-based key further down, they are not dropped."""
    lines = [l.strip() for l in raw_text.split("\n") if l.strip()]
    if not lines:
        return None
    name_clean = _re_ocr.sub(r"[^A-Za-z ]", "", lines[0]).strip().upper()
    name_clean = _re_ocr.sub(r"\s+", " ", name_clean)
    if len(name_clean) < 4:
        return None
    dob = None
    for l in lines:
        m = _re_ocr.search(r"(\d{1,2}-[A-Za-z]+-\d{4})", l)
        if m:
            dob = m.group(1); break
    return f"OCR::{name_clean}::{dob}" if dob else f"OCR::{name_clean}"

def build_ocr_patient_keys(filepaths, cache_path=OCR_CACHE_PATH):
    if os.path.exists(cache_path):
        cached = pd.read_csv(cache_path)
        cache_map = dict(zip(cached["filepath"], cached["ocr_key"]))
        print(f"Loaded {len(cache_map)} cached OCR results from {cache_path}")
    else:
        cache_map = {}
    results = {}
    to_process = [fp for fp in filepaths if fp not in cache_map]
    print(f"OCR-ing {len(to_process)} new images (cached: {len(filepaths) - len(to_process)})...")
    for i, fp in enumerate(to_process):
        try:
            key = normalize_ocr_to_patient_key(ocr_header_text(fp))
        except Exception:
            key = None
        cache_map[fp] = key if key else ""
        if (i + 1) % 200 == 0:
            print(f"  {i+1}/{len(to_process)} done")
    pd.DataFrame({"filepath": list(cache_map.keys()), "ocr_key": list(cache_map.values())}).to_csv(cache_path, index=False)
    return {fp: (cache_map[fp] if cache_map[fp] else None) for fp in filepaths}

def build_manifest(df):
    rows = []
    for _, r in df[df["is_image_ext"]].iterrows():
        try:
            with open(r["filepath"], "rb") as fh:
                data = fh.read()
            md5 = hashlib.md5(data).hexdigest()
            with Image.open(r["filepath"]) as im:
                w, h = im.size; mode = im.mode
        except Exception:
            continue
        rows.append(dict(**r, width=w, height=h, mode=mode, md5=md5))
    return pd.DataFrame(rows)

manifest = build_manifest(raw_files)
print(f"Usable images: {len(manifest)} / {len(raw_files)}")

# OCR every image's header for its true patient identity. This is the slow step (~0.3-0.4s/image);
# results are cached to OCR_CACHE_PATH so re-running the notebook after a restart does not re-OCR.
ocr_keys = build_ocr_patient_keys(manifest["filepath"].tolist())
manifest["ocr_patient_key"] = manifest["filepath"].map(ocr_keys)

n_ocr_failed = manifest["ocr_patient_key"].isna().sum()
print(f"\nOCR succeeded for {len(manifest) - n_ocr_failed} / {len(manifest)} images "
      f"({n_ocr_failed} failed -- these fall back to a per-file unique key, i.e. treated as their "
      f"own singleton patient, which is the SAFE failure mode: it can only over-split, never merge "
      f"two different real patients by accident, unlike the old filename-index scheme.)")

# fallback for OCR failures: a unique key per file, so failures never accidentally merge patients
manifest["patient_key"] = manifest["ocr_patient_key"].fillna(
    "SINGLETON::" + manifest["filepath"])

# still union any exact byte-duplicates into the same patient (belt-and-suspenders -- if the
# center's export genuinely re-exports one slice identically under two different sessions, those
# should merge even if OCR reads them slightly differently due to compression/rendering noise)
parent = {}
def find(x):
    while parent.get(x, x) != x:
        parent[x] = parent.get(parent[x], parent[x]); x = parent[x]
    return x
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb: parent[ra] = rb

for fp in manifest["filepath"]:
    parent.setdefault(fp, fp)
for pkey, sub in manifest.groupby("patient_key"):
    paths = sub["filepath"].tolist()
    for p in paths[1:]:
        union(paths[0], p)
for md5v, sub in manifest.groupby("md5"):
    paths = sub["filepath"].tolist()
    for p in paths[1:]:
        union(paths[0], p)

manifest["patient_key"] = manifest["filepath"].apply(lambda fp: manifest["patient_key"].iloc[0] if False else find(fp))
# (the line above just resolves each file to its union-find root; the actual label string doesn't
# matter beyond being a consistent group id, so we re-key by the root's own patient_key for readability)
root_to_label = manifest.set_index("filepath")["patient_key"].to_dict()
manifest["patient_key"] = manifest["filepath"].apply(lambda fp: root_to_label[find(fp)])

manifest_dedup = manifest.drop_duplicates(subset="md5", keep="first").reset_index(drop=True)
print(f"\nManifest after de-dup: {len(manifest_dedup)} images")
display(manifest_dedup.groupby("class_label").size())

print(f"\nOCR-based unique patients per class (the trustworthy count):")
display(manifest_dedup.groupby("class_label")["patient_key"].nunique())

patient_img_counts = manifest_dedup.groupby("patient_key").size().sort_values(ascending=False)
print(f"\nLargest per-patient image counts (sanity check -- should look like a normal scan-series")
print(f"count now, not one patient holding 65% of a class):")
display(patient_img_counts.head(10))


### 3.1 Spot-checking the OCR fix against the earlier bug

Directly re-checking the two specific real patients found to be wrongly merged before the OCR fix
(`SAMEIR MOH. ELBAYOUMI` from `normal_axial` and `SAMIER ASAD MOHAMED` from `normal_sagital`,
previously collapsed into one "patient" because both files happened to carry the index
`img-00031-...` in their respective, independently-numbered folders): confirm they now resolve to
two different `patient_key` values.

In [ ]:
name_check = manifest_dedup[manifest_dedup["patient_key"].str.contains("ELBAYOUMI|SAMIER ASAD", na=False, regex=True)]
display(name_check[["filename", "source_key", "patient_key"]])
print(f"\nDistinct patient_key values among these: {name_check['patient_key'].nunique()} "
      f"(should be 2, not 1 -- confirms the OCR-based key no longer merges these two people).")

print("\nFull distribution of images-per-patient (class-wise) after the OCR fix -- should look")
print("like an ordinary scan-series count per patient now, not one patient dominating a class:")
display(manifest_dedup.groupby(["class_label", "patient_key"]).size().groupby("class_label").describe())


### 3.1 PHI note (unchanged from v1, restated)

Filenames in `flair axial` / `flair sagital` carry real patient names
(`NAME_PATTERN` above). Independently of filenames, **the burned-in pixel header carries the same
PHI** (see Section 5) in *both* classes. Any public release of this dataset, code outputs, or
example figures needs those names/headers removed first, and needs a documented data-source /
consent or IRB-equivalent statement — this notebook cannot supply that statement for you.

## 4. Shortcut audit — before and after the real fix (not just border-cropping)

A trivial classifier using only global brightness/contrast/border statistics — nothing that
requires seeing anatomy — on the **raw** images:

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

def extract_shortcut_features_from_arr(arr):
    h, w = arr.shape
    b = max(1, int(0.08 * min(h, w)))
    border = np.concatenate([arr[:b, :].ravel(), arr[-b:, :].ravel(), arr[:, :b].ravel(), arr[:, -b:].ravel()])
    return [arr.mean(), arr.std(), (arr < 10).mean(), (arr > 245).mean(),
            border.mean(), border.std(), h, w]

def extract_shortcut_features_raw(fp):
    arr = np.asarray(Image.open(fp).convert("L"), dtype=np.float32)
    return extract_shortcut_features_from_arr(arr)

audit_sample = manifest_dedup.groupby("class_label", group_keys=False)[manifest_dedup.columns].apply(
    lambda d: d.sample(min(len(d), 300), random_state=SEED))

X_raw = np.array([extract_shortcut_features_raw(fp) for fp in audit_sample["filepath"]])
y_raw = (audit_sample["class_label"] == "MS (FLAIR)").astype(int).values

pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
raw_acc = cross_val_score(pipe, X_raw, y_raw, cv=cv, scoring="accuracy")
print(f"Trivial classifier accuracy on RAW images: {raw_acc.mean():.3f} +/- {raw_acc.std():.3f}")
print(f"(majority-class baseline = {100*max(y_raw.mean(), 1-y_raw.mean()):.0f}%)")


## 4.1 Checking WHY the shortcut is so hard to close: OCR the hospital name directly

Section 4 showed a large, image-statistics-only shortcut (98.8%). Rather than keep guessing at
preprocessing fixes, OCR the hospital name burned into each image's header directly (top-right
corner) and check whether it aligns with the class label -- if it does almost perfectly, no amount
of pixel-level preprocessing was ever going to fully close the gap, because the confound is in
which images got labeled which way, not in how the images were rendered.

Run on the **full** de-duplicated dataset (not a sample) with a proper 95% confidence interval --
this is the number that should be quoted in the paper, not a point estimate from ~100 images per
class. Results are cached to disk so a Kaggle session restart does not require re-OCR'ing
everything.

In [ ]:
HOSPITAL_OCR_CACHE_PATH = "hospital_ocr_cache.csv"  # cache so a restart does not re-OCR everything

def ocr_hospital_name(filepath, w_frac=0.40, h_frac=0.10):
    with Image.open(filepath) as im:
        im = im.convert("L")
        w, h = im.size
        crop = im.crop((int(w * (1 - w_frac)), 0, w, int(h * h_frac)))
        crop = crop.resize((crop.width * 3, crop.height * 3), Image.LANCZOS)
        return pytesseract.image_to_string(crop).strip()

def classify_hospital(raw_text):
    t = raw_text.upper()
    if "MANSOURA" in t:
        return "Mansoura"
    if "GOMHOR" in t:
        return "Gomhoriah"
    return "unknown"

def get_hospital_labels(filepaths, cache_path=HOSPITAL_OCR_CACHE_PATH):
    if os.path.exists(cache_path):
        cached = pd.read_csv(cache_path)
        cache_map = dict(zip(cached["filepath"], cached["hospital"]))
        print(f"Loaded {len(cache_map)} cached hospital-OCR results from {cache_path}")
    else:
        cache_map = {}
    to_process = [fp for fp in filepaths if fp not in cache_map]
    print(f"OCR-ing hospital name for {len(to_process)} new images (cached: {len(filepaths) - len(to_process)})...")
    for i, fp in enumerate(to_process):
        try:
            cache_map[fp] = classify_hospital(ocr_hospital_name(fp))
        except Exception:
            cache_map[fp] = "error"
        if (i + 1) % 200 == 0:
            print(f"  {i+1}/{len(to_process)} done")
    pd.DataFrame({"filepath": list(cache_map.keys()), "hospital": list(cache_map.values())}).to_csv(cache_path, index=False)
    return {fp: cache_map[fp] for fp in filepaths}

# FULL dataset, not a sample -- this is the number that goes in the paper.
hospital_map_full = get_hospital_labels(manifest_dedup["filepath"].tolist())
manifest_dedup["hospital"] = manifest_dedup["filepath"].map(hospital_map_full)

confound_table_full = manifest_dedup.groupby(["class_label", "hospital"]).size().unstack(fill_value=0)
print("\nClass label vs. OCR'd hospital name (FULL de-duplicated dataset, n=%d):" % len(manifest_dedup))
display(confound_table_full)

known_full = manifest_dedup[manifest_dedup["hospital"].isin(["Mansoura", "Gomhoriah"])]
agree_mask = (
    ((known_full["class_label"] == "MS (FLAIR)") & (known_full["hospital"] == "Mansoura")) |
    ((known_full["class_label"] == "Normal") & (known_full["hospital"] == "Gomhoriah"))
)
n_agree, n_known = int(agree_mask.sum()), len(known_full)
agreement_full = n_agree / n_known

# Wilson score 95% CI for the agreement proportion -- report this, not just the point estimate
from scipy.stats import norm
z = norm.ppf(0.975)
p_hat = agreement_full
denom = 1 + z**2 / n_known
center = (p_hat + z**2 / (2 * n_known)) / denom
half_width = (z * np.sqrt(p_hat * (1 - p_hat) / n_known + z**2 / (4 * n_known**2))) / denom
ci_low, ci_high = max(0, center - half_width), min(1, center + half_width)

print(f"\nFull-dataset class-label / hospital agreement: {n_agree}/{n_known} = {agreement_full:.4f} "
      f"(95% Wilson CI: [{ci_low:.4f}, {ci_high:.4f}])")
print(f"OCR failed to read a clear hospital name for {len(manifest_dedup) - n_known} / {len(manifest_dedup)} images "
      f"({(len(manifest_dedup)-n_known)/len(manifest_dedup):.1%}) -- these are excluded from the agreement rate above, "
      f"not counted as either agreeing or disagreeing.")
print("\nThis is the number to report in the paper -- computed on the full de-duplicated dataset,")
print("not a sample, with a proper confidence interval.")


## 5. The real fix, v5: crop on the UNMASKED image, mask text AFTER cropping

**Two false starts, both caught by re-running the audit rather than assuming a fix worked, and
both worth reporting in Methods as validation steps that were actually done:**

- **v2 (mask corners, then compute the crop bounding box):** 98.8% -> 98.3%. Barely moved.
- **v3 (+ per-image percentile-stretch):** 98.3% -> 92.7%/92.8%. Real progress, but still far
  above chance -- and a real Kaggle run built on this pipeline showed a NEW problem: masking the
  corner text *before* computing the brain bounding box changes how tightly the box crops,
  and it changes it **asymmetrically between classes**. Measured directly on this dataset: with
  text masked first, the average crop keeps 100% of the frame for MS/flair images but shrinks to
  only 64% of the frame for Normal images (removing the bright text corners removes the only thing
  pulling the bounding box out to the image edges for that class). That "MS looks zoomed-out,
  Normal looks zoomed-in" difference is itself a strong, trivially learnable non-anatomical
  signal -- v3's fix was inadvertently *replacing* the text-based shortcut with a new
  framing/zoom-based one.

**v5 fix:** compute the brain bounding box on the **original, unmasked** grayscale image first
(matching how the very first version of this pipeline did it, which is why an earlier, simpler
run of this notebook happened to score better on this specific audit than the "improved" v2/v3
versions -- not because it was more careful, but because leaving the text in incidentally kept
the crop consistent between classes). *Then* crop, *then* mask the four corners of the
already-cropped region (so the burned PHI is still fully removed from what the model/CLAHE/
percentile-stretch ever see), *then* percentile-stretch + CLAHE. This restores crop-frame
consistency between classes while still meeting the non-negotiable requirement of never letting
patient names/DOB/hospital text reach the model.

In [ ]:
def mask_corners(arr, w_frac=0.20, h_frac_top=0.10, h_frac_bottom=0.15):
    """Zeroes the four corners of an ALREADY-CROPPED brain region (fractions are smaller than
    v2/v3's because they are now relative to the tighter crop, not the full original frame) --
    removes burned-in PHI (patient name / DOB / hospital / WL-WW parameters) without affecting
    what counts as "brain extent" for the crop step, which must run on the unmasked image first
    (see markdown above for why -- masking before cropping created an asymmetric zoom shortcut)."""
    arr = arr.copy()
    h, w = arr.shape
    wc, ht, hb = int(w * w_frac), int(h * h_frac_top), int(h * h_frac_bottom)
    arr[:ht, :wc] = 0; arr[:ht, w - wc:] = 0
    arr[h - hb:, :wc] = 0; arr[h - hb:, w - wc:] = 0
    return arr

def crop_to_brain(im_pil, thresh=15, pad_frac=0.03):
    """Bounding-box crop computed on the UNMASKED image -- do not mask corners before this step."""
    arr = np.array(im_pil.convert("L"))
    mask = arr > thresh
    if mask.sum() == 0:
        return Image.fromarray(arr)
    ys, xs = np.where(mask)
    h, w = arr.shape
    pad_y, pad_x = int(h * pad_frac), int(w * pad_frac)
    y0, y1 = max(0, ys.min() - pad_y), min(h, ys.max() + pad_y)
    x0, x1 = max(0, xs.min() - pad_x), min(w, xs.max() + pad_x)
    cropped = arr[y0:y1, x0:x1]
    return Image.fromarray(mask_corners(cropped))  # mask AFTER crop, not before

def percentile_stretch(arr, p_low=1, p_high=99):
    brain_vals = arr[arr > 15]
    if brain_vals.size < 50:
        lo, hi = arr.min(), arr.max()
    else:
        lo, hi = np.percentile(brain_vals, p_low), np.percentile(brain_vals, p_high)
    if hi <= lo:
        hi = lo + 1
    out = np.clip((arr.astype(np.float32) - lo) / (hi - lo), 0, 1) * 255
    return out.astype(np.uint8)

def apply_clahe(arr_u8):
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    return clahe.apply(arr_u8)

def fixed_preprocess(im_pil):
    cropped = crop_to_brain(im_pil)
    arr = np.array(cropped)
    stretched = percentile_stretch(arr)
    return Image.fromarray(apply_clahe(stretched))

IMG_SIZE = 224

def extract_shortcut_features_fixed(fp, size=IMG_SIZE):
    im = fixed_preprocess(Image.open(fp)).resize((size, size), Image.BILINEAR)
    return extract_shortcut_features_from_arr(np.asarray(im, dtype=np.float32))

X_fixed = np.array([extract_shortcut_features_fixed(fp) for fp in audit_sample["filepath"]])
fixed_acc = cross_val_score(pipe, X_fixed, y_raw, cv=cv, scoring="accuracy")
print(f"Trivial classifier accuracy AFTER v5 fix (crop-then-mask, percentile-stretch, CLAHE): "
      f"{fixed_acc.mean():.3f} +/- {fixed_acc.std():.3f}  (vs {raw_acc.mean():.3f} raw)")

audit_comparison = pd.DataFrame([
    {"pipeline": "raw", "shortcut_accuracy": raw_acc.mean(), "std": raw_acc.std()},
    {"pipeline": "v2: mask-then-crop + CLAHE", "shortcut_accuracy": 0.983, "std": 0.017},
    {"pipeline": "v3: + percentile-stretch (still mask-then-crop)", "shortcut_accuracy": 0.928, "std": 0.019},
    {"pipeline": "v5: crop-then-mask + percentile-stretch + CLAHE (this fix)", "shortcut_accuracy": fixed_acc.mean(), "std": fixed_acc.std()},
])
display(audit_comparison)


**This is real, meaningful progress, but still report it honestly, not as "solved."**
On the actual dataset, this took the trivial-classifier accuracy from 98.8% (raw) to 92.8%
(v3 fix) -- a real reduction in how separable the two classes are without looking at anatomy at
all. 92.8% is still far above chance (50%), so **do not claim the confound is resolved.** What
changed and what to say in the paper:
- v1 (border-crop only, never removed the burned text): ~98.8% (no real fix)
- v2 (+ corner-mask, text removed): ~98.3% (confirms the burned text was not the main driver)
- v3 (+ per-image percentile-stretch, addressing the WL/WW rendering-window difference): ~92.8%
  (confirms the display-windowing difference *was* a major driver, though not the only one)
- The remaining gap (92.8% vs 50%) is most likely the residual framing/zoom and per-source
  resolution differences visible in Section 2 (`flair sagital` had two distinct native resolutions;
  `Normal` had one) -- interacting with how tightly `crop_to_brain`'s bounding box fits per image.
  A further, more invasive fix (e.g., forcing every image's cropped brain region to occupy a fixed
  fraction of the output canvas) was not applied here because it starts to risk fitting the
  preprocessing *to defeat this specific trivial-classifier test* rather than doing principled,
  generalizable normalization -- which would be circular and should not be done. State the 92.8%
  residual as a limitation, and lean on Sections 8/9/11 (biomarkers, fusion, ABR) as the more
  defensible evidence in the paper, exactly as the Section 0 contribution statement argues.

## 6. Patient-level, class-stratified split -- v4: capped per-patient image count

**Bug found from a real run of v3 (Kaggle, GPU) that must be fixed before any accuracy number
from this pipeline is trusted:** Section 3.1 showed one patient cluster alone holds 347 of 528
MS images (65.7% of the class). Because `split_by_patient` assigns *whole patients* to a split,
a real run put nearly all of that one patient's images into **val**, producing an image-level
split of `train: MS=146, val: MS=362` -- val had *more* MS images than train. A model can hit
apparent ~100% val accuracy almost for free against a validation set dominated by hundreds of
near-identical slices from a single scan series, with no real generalization involved.

**Fix:** cap the number of images kept per patient *before* splitting (a common, standard step in
slice-level medical-imaging pipelines specifically to prevent one patient from dominating a split
or a loss function). This also directly reduces how much of the class-level image count is
redundant near-duplicate content from a handful of patients (Section 3.1's other large clusters --
several Normal-class patients also contribute 30-50+ images each).

In [ ]:
MAX_IMAGES_PER_PATIENT = 20  # safety-net cap, not the primary fix anymore -- see note below


# Kept as a guard even after the OCR patient-identity fix (Section 3): real per-patient
# scan-series counts should now be modest (Section 3.1's describe() output), so this cap
# should rarely trigger -- it is here only in case a future data batch genuinely contains
# a patient with an unusually long series.
def cap_images_per_patient(df, group_col="patient_key", max_per_patient=MAX_IMAGES_PER_PATIENT, seed=SEED):
    rng = np.random.RandomState(seed)
    parts = []
    for pkey, sub in df.groupby(group_col):
        if len(sub) > max_per_patient:
            sub = sub.sample(max_per_patient, random_state=seed)
        parts.append(sub)
    return pd.concat(parts).reset_index(drop=True)

manifest_capped = cap_images_per_patient(manifest_dedup)
print(f"Images before cap: {len(manifest_dedup)} -> after cap ({MAX_IMAGES_PER_PATIENT}/patient): {len(manifest_capped)}")
display(manifest_capped.groupby("class_label").size())

def split_by_patient(df, group_col="patient_key", stratify_col="class_label",
                      train_size=0.70, val_size=0.15, test_size=0.15, seed=SEED):
    rng = np.random.RandomState(seed)
    train_parts, val_parts, test_parts = [], [], []
    for cls, sub in df.groupby(stratify_col):
        patients = sub[group_col].unique()
        rng.shuffle(patients)
        n = len(patients)
        n_train = max(1, int(round(n * train_size)))
        n_val = max(1, int(round(n * val_size))) if n > 2 else 0
        train_p = set(patients[:n_train]); val_p = set(patients[n_train:n_train + n_val])
        test_p = set(patients[n_train + n_val:])
        train_parts.append(sub[sub[group_col].isin(train_p)])
        val_parts.append(sub[sub[group_col].isin(val_p)])
        test_parts.append(sub[sub[group_col].isin(test_p)])
    return (pd.concat(train_parts).reset_index(drop=True), pd.concat(val_parts).reset_index(drop=True),
            pd.concat(test_parts).reset_index(drop=True))

train_df, val_df, test_df = split_by_patient(manifest_capped)
train_p, val_p, test_p = set(train_df["patient_key"]), set(val_df["patient_key"]), set(test_df["patient_key"])
assert not (train_p & val_p) and not (train_p & test_p) and not (val_p & test_p), "PATIENT LEAKAGE DETECTED"
print("Unique content-corrected patients -> train:", len(train_p), "val:", len(val_p), "test:", len(test_p))
split_counts = pd.DataFrame({"train": train_df["class_label"].value_counts(),
                              "val": val_df["class_label"].value_counts(),
                              "test": test_df["class_label"].value_counts()}).fillna(0).astype(int)
display(split_counts)
print("\n[!] Sanity check: no split's per-class image count should now be wildly out of line with")
print("    its share of patients (e.g. val should not have more images of a class than train, given")
print("    train has ~4.7x as many patients). If it still does, the cap above needs to be lower, or")
print("    another single patient is still dominating -- re-check Section 3.1 after re-running it")
print("    on the capped manifest.")


## 7. Augmentation, class imbalance, PyTorch data pipeline

*(GPU section — Kaggle/Colab.)* Same augmentation policy as v1 (train-only, no vertical flip).

In [ ]:
import random as _random
from PIL import Image as PILImage, ImageEnhance

def augment_pil(im: PILImage.Image) -> PILImage.Image:
    if _random.random() < 0.5:
        im = im.rotate(_random.uniform(-12, 12), resample=PILImage.BILINEAR, fillcolor=0)
    if _random.random() < 0.5:
        im = im.transpose(PILImage.FLIP_LEFT_RIGHT)
    if _random.random() < 0.5:
        im = ImageEnhance.Brightness(im).enhance(_random.uniform(0.85, 1.15))
        im = ImageEnhance.Contrast(im).enhance(_random.uniform(0.85, 1.15))
    if _random.random() < 0.3:
        w, h = im.size
        dx, dy = int(w * _random.uniform(-0.05, 0.05)), int(h * _random.uniform(-0.05, 0.05))
        im = im.transform(im.size, PILImage.AFFINE, (1, 0, dx, 0, 1, dy), fillcolor=0)
    return im

train_class_counts = train_df["class_label"].value_counts()
n_ms, n_normal = train_class_counts.get("MS (FLAIR)", 1), train_class_counts.get("Normal", 1)
pos_weight = n_normal / n_ms
print(f"Train class counts: {dict(train_class_counts)}  pos_weight={pos_weight:.3f}")


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

class MSMRIDatasetRGB(Dataset):
    def __init__(self, df, augment=False, img_size=IMG_SIZE):
        self.df = df.reset_index(drop=True); self.augment = augment; self.img_size = img_size
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        with Image.open(row["filepath"]) as im:
            im = fixed_preprocess(im).convert("L").resize((self.img_size, self.img_size), Image.BILINEAR)
            if self.augment: im = augment_pil(im)
        arr = np.asarray(im, dtype=np.float32) / 255.0
        rgb = np.stack([arr, arr, arr], axis=-1)
        rgb = (rgb - IMAGENET_MEAN) / IMAGENET_STD
        tensor = torch.from_numpy(rgb.transpose(2, 0, 1)).float()
        label = 1.0 if row["class_label"] == "MS (FLAIR)" else 0.0
        return tensor, torch.tensor(label, dtype=torch.float32), idx

BATCH_SIZE = 32
sample_weights = train_df["class_label"].map({"MS (FLAIR)": 1.0/n_ms, "Normal": 1.0/n_normal}).values
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_ds_rgb = MSMRIDatasetRGB(train_df, augment=True)
val_ds_rgb = MSMRIDatasetRGB(val_df, augment=False)
test_ds_rgb = MSMRIDatasetRGB(test_df, augment=False)
train_loader = DataLoader(train_ds_rgb, batch_size=BATCH_SIZE, sampler=sampler)
val_loader = DataLoader(val_ds_rgb, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds_rgb, batch_size=BATCH_SIZE, shuffle=False)
print("DataLoaders ready.")


## 8. Interpretable biomarker features, now including lesion LOCATION (McDonald-inspired, not diagnostic)

**Important scope correction, worth restating in the paper:** this dataset's label distinguishes
"a FLAIR-visible lesion is present" from "no lesion" -- it is **not** a clinical MS diagnosis.
Real MS diagnosis (McDonald criteria) additionally requires *dissemination in space* (lesions in
>=2 of: periventricular, cortical/juxtacortical, infratentorial, spinal cord) and *dissemination
in time* (new/evolving lesions across separate scans), plus ruling out other causes of white-matter
hyperintensities (small-vessel disease, migraine, NMOSD, etc.) -- none of which this 2D,
single-timepoint, single-modality dataset can establish. The features below add lesion
*location*, which is one ingredient of dissemination-in-space, as an interpretable signal --
**this remains lesion-pattern description, not diagnosis**, and every result derived from it
should be reported as such.

**Zone heuristic (2D, per-slice, approximate -- not a substitute for radiologist-read whole-brain
assessment):** for each detected lesion blob, classify it by:
- **juxtacortical**: close to the brain's outer boundary (within ~8% of an equivalent brain radius
  from the (eroded) brain-mask edge)
- **periventricular**: close to the brain's centroid (within ~30% of the equivalent radius) --
  a proxy for "near the ventricles," since true ventricle segmentation is not available here
- **infratentorial (proxy)**: only flagged for sagittal-plane images, using vertical position
  (lower/posterior region) as a coarse proxy for cerebellum/brainstem -- axial slices cannot
  support this without knowing slice height, which this 2D, non-DICOM dataset does not carry
- **deep white matter**: everything else
- (spinal cord is not assessable -- these are brain scans)

The brain mask is eroded by a few pixels before classification specifically to avoid the pial-
surface partial-volume rim being misread as "juxtacortical" by default, which was checked and
corrected during development of this notebook (an earlier, un-eroded version put ~95% of all
detections in "juxtacortical," which was a masking artifact, not a real anatomical finding).

In [ ]:
from scipy.ndimage import binary_erosion

def extract_lesion_biomarkers(im_pil, plane, z_thresh=2.0, min_lesion_px=5, size=IMG_SIZE,
                               periventricular_frac=0.30, juxtacortical_frac=0.08):
    cropped = crop_to_brain(im_pil).convert("L").resize((size, size))
    arr = np.asarray(cropped, dtype=np.float32)
    brain_mask_raw = arr > 15
    # erode a few pixels to exclude the thin pial-surface/CSF rim from counting as "juxtacortical"
    # by default -- an earlier, un-eroded version of this function put ~95% of all detections in
    # "juxtacortical" purely from that rim, which is a segmentation artifact, not a real finding.
    brain_mask = binary_erosion(brain_mask_raw, iterations=3)
    if brain_mask.sum() < 50:
        return dict(n_lesions=0, total_lesion_area=0, mean_lesion_area=0, max_lesion_area=0,
                    lesion_area_frac=0, mean_dist_center=0, mean_eccentricity=0,
                    intensity_mean=0, intensity_std=0, frac_periventricular=0,
                    frac_juxtacortical=0, frac_infratentorial=0, frac_deep_white_matter=0,
                    n_distinct_zones=0)
    dist_to_boundary = ndimage.distance_transform_edt(brain_mask)
    ys_b, xs_b = np.where(brain_mask)
    brain_cy, brain_cx = ys_b.mean(), xs_b.mean()
    brain_radius = np.sqrt(brain_mask.sum() / np.pi) + 1e-6

    brain_vals = arr[brain_mask]
    mu, sd = brain_vals.mean(), brain_vals.std() + 1e-6
    z = (arr - mu) / sd
    hyper = (z > z_thresh) & brain_mask
    lbl, n = ndimage.label(hyper)
    sizes = ndimage.sum(hyper, lbl, range(1, n + 1))
    keep = [i + 1 for i, s in enumerate(sizes) if s >= min_lesion_px]

    cy, cx = size / 2, size / 2
    dists, eccs, zones = [], [], []
    for i in keep:
        ys, xs = np.where(lbl == i)
        lcy, lcx = ys.mean(), xs.mean()
        dists.append(np.hypot(lcy - cy, lcx - cx))
        h_ext, w_ext = ys.max() - ys.min() + 1, xs.max() - xs.min() + 1
        eccs.append(max(h_ext, w_ext) / max(1, min(h_ext, w_ext)))

        dist_center_norm = np.hypot(lcy - brain_cy, lcx - brain_cx) / brain_radius
        yi, xi = int(round(lcy)), int(round(lcx))
        dist_bound_norm = (dist_to_boundary[yi, xi] / brain_radius) if brain_mask[yi, xi] else 0
        if dist_bound_norm < juxtacortical_frac:
            zone = "juxtacortical"
        elif dist_center_norm < periventricular_frac:
            zone = "periventricular"
        elif plane == "sagital" and lcy > 0.62 * size:
            zone = "infratentorial"
        else:
            zone = "deep_white_matter"
        zones.append(zone)

    areas = [sizes[i - 1] for i in keep]
    n_lesions = len(keep)
    zone_counts = Counter(zones)
    return dict(
        n_lesions=n_lesions,
        total_lesion_area=float(sum(areas)) if areas else 0,
        mean_lesion_area=float(np.mean(areas)) if areas else 0,
        max_lesion_area=float(np.max(areas)) if areas else 0,
        lesion_area_frac=float(sum(areas) / brain_mask.sum()) if areas else 0,
        mean_dist_center=float(np.mean(dists)) if dists else 0,
        mean_eccentricity=float(np.mean(eccs)) if eccs else 0,
        intensity_mean=float(mu), intensity_std=float(sd),
        frac_periventricular=zone_counts.get("periventricular", 0) / n_lesions if n_lesions else 0,
        frac_juxtacortical=zone_counts.get("juxtacortical", 0) / n_lesions if n_lesions else 0,
        frac_infratentorial=zone_counts.get("infratentorial", 0) / n_lesions if n_lesions else 0,
        frac_deep_white_matter=zone_counts.get("deep_white_matter", 0) / n_lesions if n_lesions else 0,
        # proxy for "dissemination in space": how many DISTINCT zone types have >=1 lesion in
        # THIS SINGLE SLICE. Real dissemination-in-space is a whole-brain, multi-slice, radiologist
        # judgment -- this is at best a weak, same-slice proxy for it, not a substitute.
        n_distinct_zones=len(zone_counts),
    )

FEATURE_COLS = ["n_lesions", "total_lesion_area", "mean_lesion_area", "max_lesion_area",
                 "lesion_area_frac", "mean_dist_center", "mean_eccentricity", "intensity_mean", "intensity_std",
                 "frac_periventricular", "frac_juxtacortical", "frac_infratentorial", "frac_deep_white_matter",
                 "n_distinct_zones"]

biomarker_rows = []
for _, r in manifest_capped.iterrows():  # capped manifest -- see Section 6's per-patient cap
    with Image.open(r["filepath"]) as im:
        feats = extract_lesion_biomarkers(im, plane=r["plane"])
    biomarker_rows.append({**feats, "patient_key": r["patient_key"], "class_label": r["class_label"],
                            "filepath": r["filepath"]})
biomarker_df = pd.DataFrame(biomarker_rows)
print(f"Biomarker features (incl. lesion location) computed for {len(biomarker_df)} images")
display(biomarker_df.groupby("class_label")[["n_lesions", "frac_periventricular", "frac_juxtacortical",
                                              "frac_infratentorial", "n_distinct_zones"]].mean())


In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

Xb = biomarker_df[FEATURE_COLS].values
yb = (biomarker_df["class_label"] == "MS (FLAIR)").astype(int).values
groups_b = biomarker_df["patient_key"].values

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
results = defaultdict(list)
for name, model in [("LogisticRegression", make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))),
                     ("RandomForest", RandomForestClassifier(n_estimators=300, max_depth=6, random_state=SEED))]:
    accs, aucs = [], []
    for tr_idx, te_idx in sgkf.split(Xb, yb, groups_b):
        model.fit(Xb[tr_idx], yb[tr_idx])
        prob = model.predict_proba(Xb[te_idx])[:, 1]
        accs.append(accuracy_score(yb[te_idx], prob >= 0.5))
        if len(set(yb[te_idx])) > 1: aucs.append(roc_auc_score(yb[te_idx], prob))
    results["model"].append(name); results["accuracy_mean"].append(np.mean(accs))
    results["accuracy_std"].append(np.std(accs)); results["auc_mean"].append(np.mean(aucs) if aucs else float("nan"))
biomarker_results = pd.DataFrame(results)
print("Patient-grouped 5-fold CV, biomarker features only (content-corrected patient groups):")
display(biomarker_results)


## 9. Models + Dual-Stream Fusion

**Stream A (deep):** ResNet18 / EfficientNet-B0, fine-tuned, `MAX_EPOCHS=50` + early stopping.
**Stream B (radiomics):** the 9 biomarker features from Section 8.
**Fusion:** penultimate-layer deep features concatenated with the 9 biomarker features, fed to a
classifier trained with the *same* patient-grouped CV as Section 8 — directly comparable to the
biomarker-only numbers there. This follows the fuse-then-classify pattern used for MS lesion
segmentation in the 2025 radiomics+deep-learning paper cited in Section 0, adapted here to
classification.

In [ ]:
MAX_EPOCHS = 50
EARLY_STOPPING_PATIENCE = 8

class EarlyStopping:
    def __init__(self, patience=EARLY_STOPPING_PATIENCE, min_delta=1e-4):
        self.patience = patience; self.min_delta = min_delta
        self.best_loss = float("inf"); self.counter = 0; self.should_stop = False
    def step(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss; self.counter = 0; return True
        self.counter += 1
        if self.counter >= self.patience: self.should_stop = True
        return False

def run_epoch(model, loader, criterion, optimizer=None, device="cpu"):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, total_correct, n = 0.0, 0, 0
    torch.set_grad_enabled(is_train)
    for x, y, _ in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x).squeeze(1)
        loss = criterion(logits, y)
        if is_train:
            optimizer.zero_grad(); loss.backward(); optimizer.step()
        total_loss += loss.item() * x.size(0)
        total_correct += ((torch.sigmoid(logits) >= 0.5).float() == y).sum().item()
        n += x.size(0)
    torch.set_grad_enabled(True)
    return total_loss / n, total_correct / n

def train_and_track(model, train_loader, val_loader, criterion, optimizer, max_epochs, checkpoint_path, device="cpu"):
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    stopper = EarlyStopping()
    for epoch in range(1, max_epochs + 1):
        tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = run_epoch(model, val_loader, criterion, None, device)
        history["train_loss"].append(tr_loss); history["train_acc"].append(tr_acc)
        history["val_loss"].append(val_loss); history["val_acc"].append(val_acc)
        improved = stopper.step(val_loss)
        if improved: torch.save(model.state_dict(), checkpoint_path)
        print(f"epoch {epoch:3d}/{max_epochs} | train_loss {tr_loss:.4f} acc {tr_acc:.3f} "
              f"| val_loss {val_loss:.4f} acc {val_acc:.3f}{'  <- best' if improved else ''}")
        if stopper.should_stop:
            print(f"Early stopping at epoch {epoch}."); break
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    return model, history

def build_transfer_model(arch, freeze_backbone=True, dropout_p=0.5):
    """v4: adds a Dropout layer before the final linear head (neither v1-v3 had one), which
    directly targets the near-instant Phase-2 memorization observed in a real run (val_loss
    reaching ~0.01 within one epoch of unfreezing). Dropout alone will not fix a genuine
    non-anatomical shortcut (Section 5 -- the residual ~92.7% trivial-classifier accuracy is a
    property of the DATA, not the model), but it is still necessary: without it, the model has
    no obstacle to memorizing whatever separates the classes, shortcut or not, in a single epoch."""
    from torchvision.models import resnet18, ResNet18_Weights, efficientnet_b0, EfficientNet_B0_Weights
    if arch == "resnet18":
        m = resnet18(weights=ResNet18_Weights.DEFAULT)
        m.fc = torch.nn.Sequential(torch.nn.Dropout(dropout_p), torch.nn.Linear(m.fc.in_features, 1))
    elif arch == "efficientnet_b0":
        m = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
        in_f = m.classifier[1].in_features
        m.classifier = torch.nn.Sequential(torch.nn.Dropout(dropout_p), torch.nn.Linear(in_f, 1))
    else:
        raise ValueError(arch)
    if freeze_backbone:
        for name, p in m.named_parameters():
            if "fc" not in name and "classifier" not in name: p.requires_grad = False
    return m

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

def unfreeze_last_block_only(model, arch):
    """v4: Phase 2 no longer unfreezes the ENTIRE backbone. Only the last conv block (layer4 for
    ResNet18, the last feature block for EfficientNet-B0) plus the head are trainable -- this cuts
    the number of trainable parameters drastically, which is the single biggest lever against a
    model memorizing a ~100-patient dataset within one epoch. Earlier layers keep their ImageNet
    weights, which encode generic low-level features (edges/textures) that do not need
    re-learning on a dataset this small anyway."""
    for p in model.parameters():
        p.requires_grad = False
    if arch == "resnet18":
        for p in model.layer4.parameters(): p.requires_grad = True
        for p in model.fc.parameters(): p.requires_grad = True
    elif arch == "efficientnet_b0":
        for p in model.features[-1].parameters(): p.requires_grad = True
        for p in model.classifier.parameters(): p.requires_grad = True
    return model

def train_transfer_model(arch, max_epochs=MAX_EPOCHS, phase1_frac=0.2):
    set_all_seeds(SEED)
    model = build_transfer_model(arch, freeze_backbone=True).to(device)
    pos_w = torch.tensor(pos_weight, dtype=torch.float32).to(device)
    crit = torch.nn.BCEWithLogitsLoss(pos_weight=pos_w)
    phase1_epochs = max(5, int(max_epochs * phase1_frac))
    opt1 = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=1e-3, weight_decay=1e-4)
    model, hist1 = train_and_track(model, train_loader, val_loader, crit, opt1, phase1_epochs, f"{arch}_phase1_best.pt", device)

    model = unfreeze_last_block_only(model, arch)  # v4: partial unfreeze, not the whole backbone
    phase2_epochs = max_epochs - len(hist1["train_loss"])
    opt2 = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=1e-5, weight_decay=1e-3)  # v4: 10x stronger weight_decay
    model, hist2 = train_and_track(model, train_loader, val_loader, crit, opt2, phase2_epochs, f"{arch}_best.pt", device)
    history = {k: hist1[k] + hist2[k] for k in hist1}
    return model, history

resnet_model, resnet_history = train_transfer_model("resnet18")


In [ ]:
effnet_model, effnet_history = train_transfer_model("efficientnet_b0")


### 9.1 Extract penultimate-layer deep features for fusion

In [ ]:
@torch.no_grad()
def extract_deep_features(model, arch, df, device=device):
    """Returns (N, D) deep feature matrix in the same row order as df, by hooking the
    penultimate layer (before the final linear head)."""
    model.eval()
    feats_out = []
    def hook(module, inp, out):
        feats_out.append(out.detach().cpu().numpy().reshape(out.size(0), -1))
    if arch == "resnet18":
        handle = model.avgpool.register_forward_hook(hook)
    elif arch == "efficientnet_b0":
        handle = model.avgpool.register_forward_hook(hook)
    else:
        raise ValueError(arch)
    ds = MSMRIDatasetRGB(df, augment=False)
    loader = DataLoader(ds, batch_size=64, shuffle=False)
    for x, y, idx in loader:
        model(x.to(device))
    handle.remove()
    return np.concatenate(feats_out, axis=0)

deep_feats_resnet = extract_deep_features(resnet_model, "resnet18", manifest_capped)
deep_feats_effnet = extract_deep_features(effnet_model, "efficientnet_b0", manifest_capped)
print("ResNet18 deep feature shape:", deep_feats_resnet.shape)
print("EfficientNet-B0 deep feature shape:", deep_feats_effnet.shape)


### 9.2 Fusion classifier vs. each stream alone (patient-grouped, apples-to-apples with Section 8)

In [ ]:
from sklearn.decomposition import PCA

def evaluate_stream(X, y, groups, name, n_splits=5, pca_dim=None):
    if pca_dim is not None and X.shape[1] > pca_dim:
        X = PCA(n_components=pca_dim, random_state=SEED).fit_transform(X)
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    accs, aucs = [], []
    for tr_idx, te_idx in sgkf.split(X, y, groups):
        clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=0.5))
        clf.fit(X[tr_idx], y[tr_idx])
        prob = clf.predict_proba(X[te_idx])[:, 1]
        accs.append(accuracy_score(y[te_idx], prob >= 0.5))
        if len(set(y[te_idx])) > 1: aucs.append(roc_auc_score(y[te_idx], prob))
    return {"stream": name, "accuracy_mean": np.mean(accs), "accuracy_std": np.std(accs),
            "auc_mean": np.mean(aucs) if aucs else float("nan")}

y_all = (manifest_capped["class_label"] == "MS (FLAIR)").astype(int).values
groups_all = manifest_capped["patient_key"].values
X_radiomics = biomarker_df[FEATURE_COLS].values  # same row order as manifest_capped (both built from it)

# reduce deep features to a manageable dimensionality before fusing with 9 radiomics features,
# so the radiomics signal isn't drowned out by hundreds/thousands of deep dims
X_deep_resnet_reduced = PCA(n_components=min(20, deep_feats_resnet.shape[0]-1), random_state=SEED).fit_transform(deep_feats_resnet)
X_fusion_resnet = np.concatenate([X_deep_resnet_reduced, X_radiomics], axis=1)

fusion_results = pd.DataFrame([
    evaluate_stream(X_radiomics, y_all, groups_all, "Radiomics only (Section 8 result, re-derived here)"),
    evaluate_stream(deep_feats_resnet, y_all, groups_all, "Deep (ResNet18) only", pca_dim=20),
    evaluate_stream(X_fusion_resnet, y_all, groups_all, "Fusion: Deep(ResNet18, PCA-20) + Radiomics"),
])
display(fusion_results)


**Report whichever way this goes.** If fusion beats both single streams, that supports the
2025 radiomics+DL paper's fusion hypothesis on this dataset/task too. If it doesn't clearly beat
the stronger single stream, that is *also* a legitimate, citable finding for a small-N dataset like
this one (fusion adds parameters/dimensionality without necessarily adding independent signal when
patient count is this low) — do not cherry-pick a favorable random seed or fold split to report;
the numbers above already come from the same `StratifiedGroupKFold(random_state=SEED)` used
throughout this notebook.

## 10. Standard training curves + test-set evaluation (ResNet18 / EfficientNet-B0)

In [ ]:
def to_tidy(name, hist):
    n = len(hist["train_loss"])
    return pd.DataFrame({"model": name, "epoch": range(1, n+1), "train_loss": hist["train_loss"],
                          "train_acc": hist["train_acc"], "val_loss": hist["val_loss"], "val_acc": hist["val_acc"]})

all_histories = pd.concat([to_tidy("ResNet18", resnet_history), to_tidy("EfficientNet-B0", effnet_history)]).reset_index(drop=True)
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for col, name in enumerate(all_histories["model"].unique()):
    sub = all_histories[all_histories["model"] == name]
    axes[0, col].plot(sub["epoch"], sub["train_loss"], label="train"); axes[0, col].plot(sub["epoch"], sub["val_loss"], label="val")
    axes[0, col].set_title(f"{name} - loss"); axes[0, col].legend()
    axes[1, col].plot(sub["epoch"], sub["train_acc"], label="train"); axes[1, col].plot(sub["epoch"], sub["val_acc"], label="val")
    axes[1, col].set_title(f"{name} - accuracy"); axes[1, col].legend()
plt.tight_layout(); plt.show()
print("Epochs run before early stopping (of MAX_EPOCHS=50):")
display(all_histories.groupby("model")["epoch"].max())


In [ ]:
from sklearn.metrics import (precision_score, recall_score, f1_score, average_precision_score,
                             roc_curve, precision_recall_curve, confusion_matrix, classification_report)

@torch.no_grad()
def get_predictions(model, loader, device=device):
    model.eval(); ys, ps = [], []
    for x, y, _ in loader:
        prob = torch.sigmoid(model(x.to(device)).squeeze(1)).cpu().numpy()
        ys.append(y.numpy()); ps.append(prob)
    return np.concatenate(ys), np.concatenate(ps)

predictions = {"ResNet18": get_predictions(resnet_model, test_loader),
               "EfficientNet-B0": get_predictions(effnet_model, test_loader)}

def compute_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    out = {"accuracy": accuracy_score(y_true, y_pred), "precision": precision_score(y_true, y_pred, zero_division=0),
           "recall": recall_score(y_true, y_pred, zero_division=0), "f1": f1_score(y_true, y_pred, zero_division=0)}
    if len(set(y_true)) > 1:
        out["roc_auc"] = roc_auc_score(y_true, y_prob); out["pr_auc"] = average_precision_score(y_true, y_prob)
    return out

def bootstrap_ci_metrics(y_true, y_prob, n_bootstrap=2000, seed=SEED):
    rng = np.random.RandomState(seed); n = len(y_true); rows = []
    point = compute_metrics(y_true, y_prob)
    for metric in point:
        vals = []
        for _ in range(n_bootstrap):
            idx = rng.randint(0, n, n)
            if len(set(y_true[idx])) < 2 and metric in ("roc_auc", "pr_auc"): continue
            vals.append(compute_metrics(y_true[idx], y_prob[idx]).get(metric))
        vals = [v for v in vals if v is not None]
        lo, hi = np.percentile(vals, [2.5, 97.5]) if vals else (np.nan, np.nan)
        rows.append({"metric": metric, "value": point[metric], "ci_low": lo, "ci_high": hi})
    return pd.DataFrame(rows)

for name, (y_true, y_prob) in predictions.items():
    print(f"=== {name} (test set, n={len(y_true)}) ===")
    display(bootstrap_ci_metrics(y_true, y_prob))


### 10.1 Per-image probability output (with a caveat that must travel with every number)

The model already produces a continuous score before thresholding (`torch.sigmoid(logits)`) --
below is that score reported directly as "MS probability / Normal probability" per image, rather
than only a hard MS/Normal decision.

**This is not a diagnostic confidence level, and must never be presented as one.** Given Section
0/4.1's finding (class label is ~100% confounded with acquisition site), a score of "MS: 87%"
is equally well described as "Mansoura-source-style: 87%" -- turning the decision into a
probability changes its precision, not what it is evidence of. Report these numbers, if at all, as
"model score" or "pipeline output," and repeat the confound caveat next to them -- do not let a
percentage sign imply a level of medical certainty this pipeline cannot support.

In [ ]:
@torch.no_grad()
def get_predictions_with_probs(model, loader, filepaths_ordered, device=device):
    """Returns a per-image DataFrame with both classes' scores. filepaths_ordered must match the
    loader's iteration order (i.e. shuffle=False), which is already how test_loader is built."""
    model.eval()
    rows = []
    idx_ptr = 0
    for x, y, batch_idx in loader:
        prob_ms = torch.sigmoid(model(x.to(device)).squeeze(1)).cpu().numpy()
        for bi, p_ms, y_true in zip(batch_idx.numpy(), prob_ms, y.numpy()):
            rows.append({
                "filepath": filepaths_ordered[bi],
                "true_label": "MS (FLAIR)" if y_true == 1 else "Normal",
                "model_score_MS_pct": round(float(p_ms) * 100, 1),
                "model_score_Normal_pct": round(float(1 - p_ms) * 100, 1),
                "predicted_label": "MS (FLAIR)" if p_ms >= 0.5 else "Normal",
            })
    return pd.DataFrame(rows)

test_filepaths = test_df["filepath"].tolist()  # test_ds/test_loader were built with shuffle=False from test_df
resnet_scores_df = get_predictions_with_probs(resnet_model, test_loader, test_filepaths)

print("Example per-image outputs (ResNet18, test set) -- 'model_score', NOT a diagnostic probability:")
display(resnet_scores_df.head(10))
print("\n[!] Repeat, because this is the number most likely to be misread later: a score of")
print("    'MS: 87%%' here should be read as 'the pipeline's learned pattern matched 87%% toward")
print("    the MS-labeled group' -- which, given Section 4.1's near-total site confound, cannot")
print("    currently be distinguished from 'this looks 87%% like a Mansoura-hospital export.'")


## 10.1 Label-shuffle sanity check (distinguishes "residual shortcut" from "leakage")

If Section 10's test accuracy is still suspiciously high after the v4 fixes (capped per-patient
images, dropout, partial unfreeze, stronger weight_decay), this is the standard diagnostic to run
before trusting -- or further chasing -- the number: retrain the *same* model/pipeline with the
training labels randomly shuffled (so there is, by construction, no real signal -- anatomical or
shortcut -- left to learn). Two possible outcomes and what each means:
- **Shuffled-label accuracy ~50% (chance):** the pipeline itself has no leakage. Any high accuracy
  on real labels is coming from a genuine (if unwanted) systematic difference between the classes
  in the data itself -- i.e., the residual shortcut quantified in Section 5, not a bug in the
  train/val/test mechanics.
- **Shuffled-label accuracy still well above 50%:** something in the pipeline leaks the label
  directly (e.g., a remaining patient-overlap between splits, or an image-preprocessing step that
  accidentally correlates with the label some other way). If this happens, the fixes above are not
  enough and the split/data pipeline itself needs another audit pass before any accuracy number
  from this notebook can be reported.

In [ ]:
def train_transfer_model_shuffled_labels(arch, max_epochs=15):
    """Same training pipeline as train_transfer_model, but with train AND val labels randomly
    permuted first. Kept short (15 epochs, no early-stopping patience beyond default) since the
    only thing being checked is whether accuracy rises meaningfully above chance -- if it does
    even briefly, that already answers the question."""
    set_all_seeds(SEED)
    shuffled_train_df = train_df.copy()
    shuffled_val_df = val_df.copy()
    rng = np.random.RandomState(SEED)
    shuffled_train_df["class_label"] = rng.permutation(shuffled_train_df["class_label"].values)
    shuffled_val_df["class_label"] = rng.permutation(shuffled_val_df["class_label"].values)

    shuf_train_loader = DataLoader(MSMRIDatasetRGB(shuffled_train_df, augment=True),
                                    batch_size=BATCH_SIZE, shuffle=True)
    shuf_val_loader = DataLoader(MSMRIDatasetRGB(shuffled_val_df, augment=False),
                                  batch_size=BATCH_SIZE, shuffle=False)

    model = build_transfer_model(arch, freeze_backbone=True).to(device)
    crit = torch.nn.BCEWithLogitsLoss()
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=1e-3, weight_decay=1e-4)
    model, hist = train_and_track(model, shuf_train_loader, shuf_val_loader, crit, opt,
                                   max_epochs, f"{arch}_shuffled_check.pt", device)
    return model, hist

print("Running label-shuffle sanity check (ResNet18, frozen backbone, shuffled labels)...")
_, shuffled_history = train_transfer_model_shuffled_labels("resnet18")
best_shuffled_val_acc = max(shuffled_history["val_acc"])
print(f"\nBest shuffled-label val accuracy reached: {best_shuffled_val_acc:.3f}")
print("Expected if the pipeline is leakage-free: close to 0.5. If this is well above 0.5,")
print("stop and re-audit the split/pipeline before trusting Section 10's real-label numbers.")


## 11. Grad-CAM + class-split, bootstrap-CI Attention-to-Brain-tissue Ratio (ABR)

Same construction as v1's Section 11, with the corrected corner-masked preprocessing feeding the
model (so the CAM is being computed on inputs that no longer contain burned text) and citing the
2025 Lung Attention Ratio (LAR) paper as the precedent for this metric family (Section 0, point 3).

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model; self.activations = None; self.gradients = None
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)
    def _save_activation(self, module, inp, out): self.activations = out.detach()
    def _save_gradient(self, module, grad_in, grad_out): self.gradients = grad_out[0].detach()
    def __call__(self, x):
        self.model.zero_grad()
        logit = self.model(x).squeeze(1)
        prob = torch.sigmoid(logit)
        logit.backward(torch.ones_like(logit))
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = torch.relu((weights * self.activations).sum(dim=1, keepdim=True))
        cam = torch.nn.functional.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, prob.item()

def border_mask_fn(size=IMG_SIZE, frac=0.08):
    m = np.zeros((size, size), dtype=bool); b = int(size * frac)
    m[:b, :] = m[-b:, :] = m[:, :b] = m[:, -b:] = True
    return m

def bootstrap_mean_ci(values, n_bootstrap=2000, seed=SEED):
    rng = np.random.RandomState(seed); values = np.asarray(values); n = len(values)
    means = [values[rng.randint(0, n, n)].mean() for _ in range(n_bootstrap)]
    return values.mean(), np.percentile(means, 2.5), np.percentile(means, 97.5)

border_mask = border_mask_fn(); brain_mask_ref = ~border_mask
cam_targets = {"ResNet18": (resnet_model, resnet_model.layer4[-1].conv2),
               "EfficientNet-B0": (effnet_model, effnet_model.features[-1][0])}

gradcam_rows = []
for model_name, (model, layer) in cam_targets.items():
    cam = GradCAM(model, layer)
    ds = MSMRIDatasetRGB(test_df, augment=False)
    for idx in range(len(ds)):
        x, y, _ = ds[idx]
        x = x.unsqueeze(0).to(device)
        heatmap, prob = cam(x)
        y_pred = int(prob >= 0.5)
        gradcam_rows.append({"model": model_name, "idx": idx, "y_true": int(y.item()), "y_pred": y_pred,
                              "attn_in_brain_frac": float(heatmap[brain_mask_ref].mean())})
gradcam_df = pd.DataFrame(gradcam_rows)

split_rows = []
for (model_name, y_pred), sub in gradcam_df.groupby(["model", "y_pred"]):
    m, lo, hi = bootstrap_mean_ci(sub["attn_in_brain_frac"].values)
    split_rows.append({"model": model_name, "predicted_class": "MS" if y_pred == 1 else "Normal",
                        "n": len(sub), "mean_attn_in_brain": f"{m:.3f} [{lo:.3f}, {hi:.3f}]"})
display(pd.DataFrame(split_rows))


## 12. Discussion — the confound is the finding

**Restating Section 0 here since this is where a reader expects the honest final word:** this
notebook set out to build an MS-lesion classifier and audit it for shortcuts. The audit found
something stronger than a shortcut -- a near-total confound between class label and acquisition
site (Section 4.1: ~95-100% agreement between hospital and label on a real sample). That finding,
not any accuracy number, is this notebook's result.

**What this means for every number reported above:**
- Sections 8-11's accuracy/AUC/CI numbers remain in the notebook because the *methodology* around
  them (patient-grouped CV, OCR-corrected patient identity, bootstrap CIs, capped per-patient
  images) is sound and reusable -- but none of those numbers should be reported in the paper as
  evidence of disease-detection ability, because "detects MS" and "detects Mansoura-hospital-style
  images" are indistinguishable hypotheses on this data.
- The gap between the interpretable biomarker model (~90%) and the deep models (100%) is likely
  telling you the deep models found *additional* site-correlated signal beyond lesion morphology
  (compression artifacts, scanner-specific noise texture, subtle framing) that the biomarker
  features, constrained to lesion blobs inside brain tissue, cannot access -- not that the deep
  models understand MS pathology better.
- Domain-adversarial training was considered and not run, for the reason given in Section 0: it
  cannot recover a useful, debiased signal when the domain and the label are (almost) the same
  variable. This is worth stating explicitly in the paper as a considered-and-rejected approach,
  with the reasoning, rather than omitting it.

**What would actually fix this, for future data collection (not something this notebook or any
amount of further modeling can retroactively solve):** images labeled MS and Normal *from the same
sites*, ideally the same scanners, so that within-site variation in label is available for a model
(or DANN) to learn from. Short of that, any classifier trained on this specific dataset should be
evaluated only on held-out data from the *same two sites in the same proportions*, and even then
described as learning "this two-site labeling scheme," not "MS."

**Standing limitations, restated:**
- Patient counts remain small (Section 3: 82 MS patients, 34 Normal, after OCR correction) even
  before the confound issue -- CIs everywhere should be read as wide, not narrow.
- 2D slice-level analysis, not validated 3D lesion segmentation.
- PHI (names, DOB, hospital identifiers -- both in filenames and burned into pixels) must be
  removed and a documented consent/IRB-equivalent statement obtained before any public release;
  nothing in this notebook substitutes for that.

**Contribution, final version:** a worked example of auditing a real clinical-imaging dataset down
to a complete site/label confound via burned-in-metadata OCR, including a principled explanation
of why a standard debiasing technique (DANN) does not apply once a confound is total rather than
partial -- offered as a methodological case study and a cautionary template for auditing similarly
multi-source medical-imaging datasets before publishing any classification result built on them.